# Tracing 请求追踪教程 (v0.7)

本教程帮助你理解可观测性中**请求追踪**的概念和使用方式。

## 学习目标

1. 理解分布式追踪：Trace 和 Span 模型
2. 掌握嵌套 Span：父子调用关系
3. 学会错误追踪：自动捕获异常
4. 了解跨模块追踪：同一 trace_id 贯穿多个操作

In [ ]:
# Step 1: 安装依赖
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", 
    "openai", "python-dotenv", "zhipuai", "-q"
])
print("依赖安装完成！")

In [ ]:
# Step 2: 设置项目路径
import sys
from pathlib import Path

cwd = Path.cwd()
if cwd.name == "notebooks":
    project_root = cwd.parent
else:
    project_root = cwd
    while not (project_root / "pyproject.toml").exists():
        project_root = project_root.parent

src_path = project_root / "src"
sys.path.insert(0, str(src_path))

print(f"项目根目录: {project_root}")
print(f"源码路径: {src_path}")

## 1. 什么是请求追踪？

追踪（Tracing）跟踪一个请求在系统中的流转路径。

| 概念 | 说明 |
|------|------|
| **Trace** | 一次完整的请求（由 trace_id 标识） |
| **Span** | 一次操作的开始和结束（含时间） |
| **SpanEvent** | Span 内的关键节点 |
| **parent_span_id** | 父 Span，形成调用树 |

In [ ]:
# 导入 Tracing 相关类型
from rogue_rabbit.contracts.log import Span, SpanStatus, SpanEvent
from rogue_rabbit.core.log_manager import Tracer
from rogue_rabbit.runtime.log_store import InMemorySpanStore
import time

store = InMemorySpanStore()
tracer = Tracer(store=store)

print("导入成功！")

## 2. 基础追踪

使用 `start_trace()` 创建追踪，`start_span()` 创建 Span（上下文管理器自动计时）。

In [ ]:
# 创建追踪
trace_id = tracer.start_trace("agent.request", attributes={"user_id": "user1"})
print(f"trace_id: {trace_id}")

# 创建 Span
with tracer.start_span("llm.call", trace_id=trace_id) as span:
    span.add_event("request_sent", {"model": "gpt-4", "tokens": 100})
    time.sleep(0.05)
    span.add_event("response_received", {"tokens": 150})

# 查看结果
spans = tracer.get_trace(trace_id)
for s in spans:
    duration = f"{s.duration_ms:.1f}ms" if s.duration_ms else "N/A"
    print(f"  {s.name} | status={s.status.value} | duration={duration}")
    for evt in s.events:
        print(f"    event: {evt.name} {evt.attributes}")

## 3. 嵌套 Span

Span 支持嵌套，通过 `parent_span_id` 形成调用树。Tracer 自动关联当前活跃的 Span。

In [ ]:
# 新的追踪
store.clear()
tracer2 = Tracer(store=store)
trace_id = tracer2.start_trace("agent.chat")

# 嵌套调用
with tracer2.start_span("permission.check", trace_id=trace_id) as perm_span:
    perm_span.add_event("start_check", {"role": "user"})
    time.sleep(0.02)
    
    with tracer2.start_span("authorizer.check_policy") as policy_span:
        policy_span.add_event("policy_loaded", {"policy": "user-basic"})
        time.sleep(0.01)

with tracer2.start_span("llm.call", trace_id=trace_id) as llm_span:
    llm_span.add_event("request_prepared")
    
    with tracer2.start_span("llm.tokenize") as tok_span:
        time.sleep(0.01)
    
    with tracer2.start_span("llm.inference") as inf_span:
        time.sleep(0.03)
        inf_span.add_event("completed", {"output_tokens": 100})

# 展示调用树
spans = tracer2.get_trace(trace_id)
print(f"追踪树 (trace_id={trace_id}):\n")

span_map = {s.span_id: s for s in spans}
children = {}
for s in spans:
    pid = s.parent_span_id
    children.setdefault(pid, []).append(s)

def print_tree(span, indent=""):
    duration = f"{span.duration_ms:.1f}ms" if span.duration_ms else "N/A"
    print(f"{indent}{span.name} [{duration}]")
    for child in children.get(span.span_id, []):
        print_tree(child, indent + "  ")

for root in children.get(None, []):
    print_tree(root)

## 4. 错误追踪

当 Span 内发生异常时，自动标记 `status=ERROR` 并记录错误事件。

In [ ]:
store.clear()
tracer3 = Tracer(store=store)
trace_id = tracer3.start_trace("agent.request")

# 正常操作
with tracer3.start_span("session.load", trace_id=trace_id):
    time.sleep(0.01)

# 失败操作
try:
    with tracer3.start_span("tool.execute", trace_id=trace_id) as span:
        span.add_event("tool_selected", {"tool": "file_reader"})
        raise FileNotFoundError("文件不存在: /secret/data.txt")
except FileNotFoundError:
    pass

# 查看结果
spans = tracer3.get_trace(trace_id)
for s in spans:
    duration = f"{s.duration_ms:.1f}ms" if s.duration_ms else "N/A"
    print(f"  {s.name} | status={s.status.value} | duration={duration}")
    for evt in s.events:
        print(f"    event: {evt.name} {evt.attributes}")

## 5. 跨模块追踪

同一 `trace_id` 可以贯穿多个模块，实现端到端的请求追踪。

In [ ]:
store.clear()
tracer4 = Tracer(store=store)

# 模拟一个完整的 Agent 请求
trace_id = tracer4.start_trace(
    "agent.complete_request",
    attributes={"user_id": "user1", "input": "帮我写一个函数"}
)

# Step 1: 权限检查
with tracer4.start_span("permission.check", trace_id=trace_id) as span:
    time.sleep(0.01)
    span.add_event("checked", {"result": "allowed"})

# Step 2: LLM 调用
with tracer4.start_span("llm.generate", trace_id=trace_id) as llm_span:
    llm_span.add_event("request_sent", {"model": "gpt-4"})
    with tracer4.start_span("llm.api_call") as api_span:
        time.sleep(0.05)
    llm_span.add_event("response_parsed", {"output_tokens": 80})

# Step 3: 保存记忆
with tracer4.start_span("memory.save", trace_id=trace_id) as mem_span:
    time.sleep(0.01)
    mem_span.add_event("saved", {"memory_id": "mem_123"})

# 追踪摘要
spans = tracer4.get_trace(trace_id)
print(f"追踪摘要 | trace_id={trace_id}")
print(f"  总 Span 数: {len(spans)}")

# 按耗时排序
timed = [s for s in spans if s.duration_ms is not None]
timed.sort(key=lambda s: s.duration_ms, reverse=True)
print(f"\n耗时排序:")
for s in timed:
    print(f"  {s.duration_ms:7.1f}ms  {s.name}")

## 总结

### 核心概念

| 概念 | 说明 |
|------|------|
| `Span` | 一次操作（名称 + 开始/结束 + 状态） |
| `Trace` | 一组 Span 通过 trace_id 组成调用链 |
| `SpanEvent` | Span 内的关键节点 |
| `Tracer` | 追踪管理器 |

### 下一步

- 运行 `experiments/19_tracing.py`
- 继续学习 [10_debugging.ipynb](10_debugging.ipynb) 调试支持